Example trapped resonance objective function computation and optimization script

In [1]:
# Objective function computation

import desc.io
from desc.objectives import (
    TrappedResonance
)
import numpy as np
import jax.numpy as jnp

## Inputs
eq = desc.io.load("Equilibria/someQAeq.h5")
rhos = np.sqrt(np.linspace(0.1,0.9,5)) # rho = sqrt(s)
alphas = np.linspace(0,2*np.pi,5)
KE_frac = np.array([1]) 
pitch_invs = jnp.linspace(5.8,5.9,2)
N=0
##

eq_periodicity = (np.inf,np.inf,np.inf)
grid_psi = eq._get_rtz_grid( # grid to compute psi
    rhos, # radial
    np.array([0]), # poloidal (alpha in this case)
    np.array([0]), # toroidal (zeta in this case)
    coordinates="raz", # rho, alpha, zeta input coordinates
    period=eq_periodicity, # periodicity of coordinate (rho,alpha,zeta)
)
Psi = eq.compute("Psi",grid=grid_psi)
obj = TrappedResonance(eq,rho=rhos,pitch_invs=pitch_invs,KE_frac=KE_frac,alpha=alphas,N=N,Psi=Psi['Psi']) # grid for objective function is set in init of TrappedResonance class
obj.build()
value = obj.compute(eq.params_dict)
print(value)

Precomputing transforms


In [1]:
# Example optimization

import sys
import os
import math
import jax.numpy as jnp

sys.path.insert(0, os.path.abspath("."))
sys.path.append(os.path.abspath("../../../"))
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 20

import desc.io
from desc.grid import LinearGrid, ConcentricGrid
from desc.objectives import (
    ObjectiveFunction,
    FixBoundaryR,
    FixBoundaryZ,
    FixPressure,
    FixIota,
    FixPsi,
    ForceBalance,
    QuasisymmetryTripleProduct,
    GammaC,
    TrappedResonance
)
from desc.optimize import Optimizer


# load initial equilibrium
eq_init = desc.io.load("Equilibria/desc_eq_beta2.5_QA.h5") # QA
eq_init.change_resolution(L=4,M=4,N=4) # change resolution to make optimization faster

# Specify equilibrium nfp and helicity
N = 0 # QA
nfp = eq_init.NFP

# optimizer = Optimizer("proximal-scipy-bfgs")
optimizer = Optimizer("proximal-lsq-exact")

# indices of boundary modes we want to optimize
idx_Rcc = eq_init.surface.R_basis.get_idx(M=1, N=2)
idx_Rss = eq_init.surface.R_basis.get_idx(M=-1, N=-2)
idx_Zsc = eq_init.surface.Z_basis.get_idx(M=-1, N=2)
idx_Zcs = eq_init.surface.Z_basis.get_idx(M=1, N=-2)
print("surface.R_basis.modes is an array of [l,m,n] of the surface modes:")
print(eq_init.surface.R_basis.modes[0:10])

# boundary modes to constrain
R_modes = np.delete(eq_init.surface.R_basis.modes, [idx_Rcc, idx_Rss], axis=0)
Z_modes = np.delete(eq_init.surface.Z_basis.modes, [idx_Zsc, idx_Zcs], axis=0)

eq_qs_T = eq_init.copy()  # make a copy of the original equilibria

constraints = (
    ForceBalance(eq=eq_qs_T),  # enforce JxB-grad(p)=0 during optimization
    FixBoundaryR(eq=eq_qs_T, modes=R_modes),  # fix specified R boundary modes
    FixBoundaryZ(eq=eq_qs_T, modes=Z_modes),  # fix specified Z boundary modes
    FixPressure(eq=eq_qs_T),  # fix pressure profile
    FixIota(eq=eq_qs_T),  # fix rotational transform profile
    FixPsi(eq=eq_qs_T),  # fix total toroidal magnetic flux
)

# Create grid for QSTripleProduct objective 
grid_vol = ConcentricGrid(
    L=eq_init.L_grid,
    M=eq_init.M_grid,
    N=eq_init.N_grid,
    NFP=eq_init.NFP,
    sym=eq_init.sym,
)

# Initializations for TrappedResonance objective
rhos = np.sqrt(np.linspace(0.1,0.9,5)) # rho = sqrt(s)
alphas = np.linspace(0,2*np.pi,5)
KE_frac = np.array([1]) 
pitch_invs = jnp.linspace(5.8,5.9,2)
eq_periodicity = (np.inf,np.inf,np.inf) # periodicity in zeta for these equilibrium to make rtz grid
grid_psi = eq_qs_T._get_rtz_grid( # grid to compute psi
    rhos, # radial
    np.array([0]), # poloidal (alpha in this case)
    np.array([0]), # toroidal (zeta in this case)
    coordinates="raz", # rho, alpha, zeta input coordinates
    period=eq_periodicity, # periodicity of coordinate (rho,alpha,zeta)
)
Psi = eq_qs_T.compute("Psi",grid=grid_psi)

# Objective for resonance
objective_fT = ObjectiveFunction(
    (
        QuasisymmetryTripleProduct(eq=eq_qs_T, grid=grid_vol),
        TrappedResonance(eq=eq_qs_T,rho=rhos,pitch_invs=pitch_invs,KE_frac=KE_frac,alpha=alphas,N=N,Psi=Psi['Psi'])
    ), 
)

eq_qs_T, result_T = eq_qs_T.optimize(
    objective=objective_fT,
    constraints=constraints,
    optimizer=optimizer,
    ftol=5e-2,  # stopping tolerance on the function value
    xtol=1e-6,  # stopping tolerance on the step size
    gtol=1e-6,  # stopping tolerance on the gradient
    maxiter=10,  # maximum number of iterations
    options={
        "perturb_options": {"order": 2, "verbose": 0},  # use 2nd-order perturbations
        "solve_options": {
            "ftol": 5e-3,
            "xtol": 1e-6,
            "gtol": 1e-6,
            "verbose": 0,
        },  # for equilibrium subproblem
    },
    copy=False,  # copy=False we will overwrite the eq_qs_T object with the optimized result
    verbose=3,
)

/Users/paullab/codes/DESC_fork_08282025/DESC_TrappedRes/desc/utils.py:572: UserWarning: Reducing radial (L) resolution can make plasma boundary inconsistent. Recommend calling `eq.surface = eq.get_surface_at(rho=1.0)`
  warnings.warn(msg, err)


surface.R_basis.modes is an array of [l,m,n] of the surface modes:
[[ 0 -4 -4]
 [ 0 -3 -4]
 [ 0 -2 -4]
 [ 0 -1 -4]
 [ 0 -4 -3]
 [ 0 -3 -3]
 [ 0 -2 -3]
 [ 0 -1 -3]
 [ 0 -4 -2]
 [ 0 -3 -2]]
Building objective: QS triple product
Precomputing transforms
Timer: Precomputing transforms = 474 ms
Building objective: TrappedResonance
Precomputing transforms
Timer: Precomputing transforms = 507 ms
Timer: Objective build = 1.96 sec
Building objective: force
Precomputing transforms
Timer: Precomputing transforms = 145 ms
Timer: Objective build = 176 ms
Timer: Objective build = 862 us
Timer: Eq Update LinearConstraintProjection build = 1.79 sec
Timer: Proximal projection build = 5.12 sec
Building objective: lcfs R
Building objective: lcfs Z
Building objective: fixed pressure
Building objective: fixed iota
Building objective: fixed Psi
Timer: Objective build = 334 ms
Timer: LinearConstraintProjection build = 586 ms
Number of parameters: 4
Number of objectives: 12549
Timer: Initializing the optimizat